# 11.7 - Chunking

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

LLMs and embedding models have input limits, and a 10-page document rarely answers a specific question - a paragraph does. Chunking splits documents into retrieval-sized segments that each carry one idea, enough context, and a reference back to the source. Bad chunking is the #1 cause of poor RAG quality.

## 2. Why Does This Matter?

Chunk size and overlap determine both retrieval precision and how much context each hit carries. Getting this wrong silently caps system quality.

## 3. Prerequisites

Unit 11.6 (Document Ingestion).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Use RecursiveCharacterTextSplitter with configurable size and overlap
- See how chunk_size and chunk_overlap change the output
- Seed metadata (source, page) on every chunk and print before/after

## 5. Mental Model

Chunking is cutting a book into flashcards: each card holds one complete idea, enough context, and a reference to where it came from.

```text
Document -> Split Strategy -> Chunks + Metadata -> (next: Embedding)
```


## 6. Setup
`RecursiveCharacterTextSplitter` from `langchain_text_splitters` splits by its separator list in order - breaking on paragraph breaks first, then newlines, sentences, words - producing natural-boundary chunks.

In [1]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    print("langchain_text_splitters available")
except Exception as e:
    RecursiveCharacterTextSplitter = None
    print("missing:", type(e).__name__)


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


langchain_text_splitters available


## 7. Chunk Size vs Overlap
Smaller chunks = more precise but less context; larger chunks = more context but noisier. Overlap preserves sentences that straddle a boundary.

In [2]:
text = """The return policy allows returns within 30 days of purchase.
Items must be in original packaging and accompanied by a receipt.
Shipping costs are non-refundable.
Contact support@example.com to initiate a return.
Refunds are processed within 5 to 7 business days.
All returned items are inspected for damage before approval."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_text(text)
print(f"orig({len(text)} chars) -> {len(chunks)} chunks of ~120 chars")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c)
    print()


orig(323 chars) -> 4 chunks of ~120 chars
--- chunk 0 (60 chars) ---
The return policy allows returns within 30 days of purchase.

--- chunk 1 (100 chars) ---
Items must be in original packaging and accompanied by a receipt.
Shipping costs are non-refundable.

--- chunk 2 (84 chars) ---
Shipping costs are non-refundable.
Contact support@example.com to initiate a return.

--- chunk 3 (111 chars) ---
Refunds are processed within 5 to 7 business days.
All returned items are inspected for damage before approval.



## 8. Compare Sizes
A quick sweep over chunk sizes on the same text shows the count and average length you get. This tells you how many vectors you'll index.

In [3]:
for size in (80, 120, 200):
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 5,
                                        separators=["\n\n", "\n", ". ", " ", ""])
    cs = sp.split_text(text)
    avg_len = sum(len(c) for c in cs) / len(cs)
    print(f"chunk_size={size:3d} -> {len(cs)} chunks, avg {avg_len:.0f} chars")


chunk_size= 80 -> 6 chunks, avg 53 chars
chunk_size=120 -> 4 chunks, avg 80 chars
chunk_size=200 -> 2 chunks, avg 178 chars


## 9. Metadata Propagation + A Powered Document
Now we go a step up: use `create_documents` to split many docs at once, attaching `source` and `page` metadata to **every chunk**. That metadata becomes critical at retrieval time for citation and filtering.

In [4]:
docs = [
    {"text": "Section A. The liability clause limits damages to the purchase price.", "source": "policy.pdf", "page": 2},
    {"text": "Section B. Parties agree to arbitration within 30 days of a dispute.", "source": "policy.pdf", "page": 3},
]
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20,
                                          separators=["\n\n", "\n", ". ", " ", ""])
# split each doc separately so we can seed its own metadata
all_chunks = []
for d in docs:
    pieces = splitter.split_text(d["text"])
    for p in pieces:
        all_chunks.append({"text": p, "metadata": {"source": d["source"], "page": d["page"]}})

print("total chunks:", len(all_chunks))
for c in all_chunks:
    print(f"meta={c['metadata']} | {c['text'][:60]}")


total chunks: 2
meta={'source': 'policy.pdf', 'page': 2} | Section A. The liability clause limits damages to the purcha
meta={'source': 'policy.pdf', 'page': 3} | Section B. Parties agree to arbitration within 30 days of a 


## 10. Before / After
Final sanity: show original doc lengths vs the chunked + metadata-seeded records.

In [5]:
print("BEFORE: 2 raw documents (whole text each)")
for d in docs:
    print(f"   {d['source']} page {d['page']}: {len(d['text'])} chars")
print("AFTER: per-chunk records with metadata")
for c in all_chunks:
    print(f"   {c['metadata']['source']} p.{c['metadata']['page']} ({len(c['text'])} chars)")


BEFORE: 2 raw documents (whole text each)
   policy.pdf page 2: 69 chars
   policy.pdf page 3: 68 chars
AFTER: per-chunk records with metadata
   policy.pdf p.2 (69 chars)
   policy.pdf p.3 (68 chars)



## Common Mistakes

- Chunking by arbitrary char count, splitting meaning.
- No overlap - losing context at boundaries.
- Chunks too small (lose context) or too large (dilute relevance).
- Not propagating source/page metadata.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Relevant answer split across chunks | Too small / no overlap | Increase size + overlap |
| Chunks contain unrelated content | Too large | Reduce size, semantic chunking |
| Same sentence in 3 chunks | Overlap too large | Reduce overlap ratio |
| Table data garbled | Table split across chunks | Chunk tables separately |

## Best Practices

- Start with recursive character splitting.
- Use 10-20% overlap.
- Always propagate source metadata to every chunk.
- Test on 10+ real docs before scaling.
- Measure retrieval quality at different sizes.

## Hands-On Practice

1. **Basic:** Chunk a paragraph with fixed-size splitting.
2. **Guided:** Compare fixed vs recursive splitting on the same doc.
3. **Independent:** Chunk a markdown doc, preserving headers as metadata.
4. **Realistic:** Test 3 chunk sizes and measure retrieval precision.
5. **Challenge:** Implement semantic chunking that splits at paragraph boundaries.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
